In [1]:
! pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.1/124.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.9/246.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is in

In [2]:
from fastai.vision.all import *
from fastbook import *

In [3]:
def list_methods(var):
    return [a for a in dir(var) if callable(getattr(var, a))]

In [4]:
path = untar_data(URLs.IMAGENETTE)

<div><progress max="1557161267" value="1557168128"></progress> 100.00% [1557168128/1557161267 00:25&lt;00:00]</div>

In [5]:
Path.BASE_PATH = path
path.ls().sorted()

[Path('noisy_imagenette.csv'), Path('train'), Path('val')]

In [6]:
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    get_y=parent_label,
    item_tfms=Resize(460),
    batch_tfms=aug_transforms(size=224, min_scale=0.75),
)

dls = dblock.dataloaders(path, bs=64)

In [7]:
model = xresnet50(n_out=dls.c)
learn = Learner(dls, model, loss_func=CrossEntropyLossFlat(), metrics=accuracy)
learn.fit_one_cycle(5, 3e-3)

epoch,train_loss,valid_loss,accuracy,time
0,1.587196,1.560043,0.496266,02:22
1,1.204216,1.081100,0.648618,02:32
2,0.966264,1.070755,0.660941,02:33
3,0.743226,0.689123,0.785661,02:33
4,0.613687,0.552466,0.828603,02:33


### Normalization

In [8]:
x,y = dls.one_batch()
x.shape, y.shape

(torch.Size([64, 3, 224, 224]), torch.Size([64]))

In [9]:
x.mean(dim=[0,2,3]), x.std(dim=[0,2,3])

(TensorImage([0.4675, 0.4609, 0.4204], device='cuda:0'),
 TensorImage([0.2911, 0.2801, 0.2950], device='cuda:0'))

In [10]:
imagenet_stats

([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

In [11]:
def get_dls(bs, size):
    dblock = DataBlock(
        blocks=(ImageBlock, CategoryBlock),
        get_items=get_image_files,
        get_y=parent_label,
        item_tfms=Resize(460),
        batch_tfms=[
            *aug_transforms(size=size, min_scale=0.75),
            Normalize.from_stats(*imagenet_stats)
        ]
    )

    return dblock.dataloaders(path, bs=bs)

In [12]:
dls = get_dls(64, 224)

In [13]:
x,y = dls.one_batch()
x.shape, y.shape

(torch.Size([64, 3, 224, 224]), torch.Size([64]))

In [14]:
x.mean(dim=[2,0,3]), x.std(dim=[3,2,0])

(TensorImage([-0.0620,  0.0279,  0.1135], device='cuda:0'),
 TensorImage([1.1985, 1.2326, 1.3172], device='cuda:0'))

In [15]:
model = xresnet50(dls.c)
learn = Learner(dls, model, loss_func=CrossEntropyLossFlat(), metrics=accuracy)
learn.fit_one_cycle(5, 3e-3)

Downloading: "https://s3.amazonaws.com/fast-ai-modelzoo/xrn50_940.pth" to /root/.cache/torch/hub/checkpoints/xrn50_940.pth


100%|██████████| 244M/244M [00:06<00:00, 41.0MB/s] 


epoch,train_loss,valid_loss,accuracy,time
0,1.287211,1.777086,0.507095,02:38
1,0.964464,1.975547,0.533234,02:35
2,0.740417,0.748969,0.763630,02:33
3,0.551395,0.525230,0.842420,02:34
4,0.446666,0.441021,0.854369,02:34


It doesn't always help much, but there's still value in doing this, particularly in transfer learning. If the original model was trained with data whose mean and std dev are different from the model (eg if your min value is 0), then the model is inherently doing something misaligned with what it was trained to do. So it's up to you to find out how best to tranform your data (i.e. normalize it) to the form that the model was trained on, so it can make sensible predictions which you can later reverse to the scale of your own data. Likewise, if you're publishing a model, be sure to also distribute the values of those 2 properties for others to have easy access to them.

### Progressive Resizing

In [16]:
# first train at a smaller size
dls = get_dls(128, 128)
learn = Learner(dls, xresnet50(dls.c), loss_func=CrossEntropyLossFlat(), metrics=accuracy)
learn.fit_one_cycle(4, 3e-3)

epoch,train_loss,valid_loss,accuracy,time
0,1.429031,3.001925,0.420463,01:09
1,0.943383,1.690426,0.563480,01:08
2,0.679758,0.620351,0.798730,01:08
3,0.496225,0.470657,0.849515,01:08


In [17]:
# then train at a larger size
learn.dls = get_dls(64, 224)
learn.fine_tune(5, 1e-3)

epoch,train_loss,valid_loss,accuracy,time
0,0.597917,0.656026,0.794623,02:34


epoch,train_loss,valid_loss,accuracy,time
0,0.461104,0.797519,0.773338,02:36
1,0.486716,0.634609,0.820015,02:34
2,0.410704,0.429584,0.867812,02:34
3,0.331197,0.370177,0.886856,02:34
4,0.289734,0.335868,0.900672,02:34


Very solid results obviously! Big improvement.

### Test Time Augmentation (TTA)

In [18]:
preds, targ = learn.tta()
preds, targ

epoch,train_loss,valid_loss,accuracy,time


<div></div>

(tensor([[4.1794e-03, 7.7040e-03, 6.9487e-02,  ..., 1.6422e-06, 1.3912e-06, 1.0099e-06],
         [1.7181e-06, 1.3254e-05, 8.6959e-06,  ..., 6.1730e-11, 2.4502e-11, 3.9504e-11],
         [9.9425e-01, 1.4428e-03, 1.2476e-06,  ..., 5.6946e-12, 9.7307e-12, 7.6383e-12],
         ...,
         [1.0000e+00, 5.7881e-08, 6.8872e-09,  ..., 4.5908e-16, 5.5752e-16, 3.4382e-16],
         [1.9483e-02, 3.2870e-02, 7.2916e-05,  ..., 2.3373e-08, 3.3933e-08, 2.6763e-08],
         [2.1791e-04, 1.3638e-04, 2.8195e-04,  ..., 1.7040e-08, 6.7403e-09, 6.6686e-09]]),
 tensor([4, 9, 0,  ..., 0, 3, 9]))

In [19]:
accuracy(preds, targ).item()

0.9055265188217163

TTA brought an improvement, which is what we want to see.